# Full Pipeline Validation Report

**Accelerator: GPU recommended** (CPU works but is ~10× slower)

## What this notebook measures

This notebook validates the **complete model pipeline** exactly as it runs in `run_captions.py` — not just the signs head in isolation.

| Section | What is tested |
|---|---|
| **Signs head** | Per-class top-1/3/5 accuracy, confusion pairs, confidence calibration |
| **Fingerspelling head** | Character Error Rate (CER), decoded output samples |
| **Continuous feed simulation** | Slides the real buffer window over each clip frame-by-frame exactly as the webcam does, then checks whether `CaptionState` commits the correct sign |
| **Commit latency** | How many frames into a clip the correct sign gets committed |
| **Silence rejection** | False positive rate — how often the model commits a wrong sign when it should be silent |
| **Confidence threshold sweep** | What accuracy/recall tradeoff you get at different `CONFIDENCE_THRESH` values |

## Before running: attach these datasets
| Dataset | Kaggle slug |
|---|---|
| Your project files | `asl-project` (your username) |
| ASL Signs competition | `asl-signs` (Google Isolated Sign Language Recognition) |
| ASL Fingerspelling | `asl-fingerspelling` (Google ASL Fingerspelling Recognition) |

The fingerspelling dataset is only needed for Section 2. If you don't have it attached, that section is skipped automatically and the rest of the report still runs.

In [1]:
# ── Cell 1: Resolve paths and load vocabularies ────────────────────────
# Unique marker files are used to identify each dataset reliably.
# This avoids the collision where both asl-signs and asl-fingerspelling
# have a train.csv, causing the wrong dataset to be picked.
#
# asl-signs unique marker      : train_landmark_files/ directory
# asl-fingerspelling marker    : supplemental_landmarks/ directory OR
#                                a train.csv whose columns include 'phrase'
import json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

KAGGLE_INPUT = Path('/kaggle/input')
OUT_DIR      = Path('/kaggle/working')
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Print full input listing to aid debugging
print('\n=== Attached datasets ===')
for ds in sorted(KAGGLE_INPUT.iterdir()):
    files = [p for p in ds.rglob('*') if p.is_file()]
    print(f'  [{ds.name}]  {len(files)} files')
print()

# ── Resolve PROJECT_DIR via model.py ──────────────────────────────────
model_py = next(KAGGLE_INPUT.rglob('model.py'), None)
if not model_py:
    raise FileNotFoundError('model.py not found. Attach your asl-project dataset.')
PROJECT_DIR = model_py.parent
sys.path.insert(0, str(PROJECT_DIR))
print(f'PROJECT_DIR : {PROJECT_DIR}')

# ── Resolve weights ────────────────────────────────────────────────────
weights = (next(PROJECT_DIR.rglob('asl_model_weights_v3.pt'), None) or
           next(PROJECT_DIR.rglob('asl_model_weights_v2.pt'), None))
if not weights:
    raise FileNotFoundError('No model weights (.pt) found in asl-project dataset.')
print(f'Weights     : {weights.name}  ({weights.stat().st_size/1e6:.1f} MB)')

# ── Resolve ASL Signs dataset ─────────────────────────────────────────
# Unique to asl-signs: train_landmark_files/ directory containing parquets
lm_root_candidate = next(KAGGLE_INPUT.rglob('train_landmark_files'), None)
if lm_root_candidate is None:
    # Fallback: find a train.csv that has a 'sign' column
    lm_root_candidate_dir = None
    for csv_path in KAGGLE_INPUT.rglob('train.csv'):
        try:
            cols = pd.read_csv(csv_path, nrows=1).columns.tolist()
            if 'sign' in cols:
                lm_root_candidate_dir = csv_path.parent
                break
        except Exception:
            continue
    if lm_root_candidate_dir is None:
        raise FileNotFoundError(
            'ASL Signs dataset not found.\n'
            'Add data → Competitions → Google Isolated Sign Language Recognition (asl-signs)'
        )
    ASL_SIGNS_DIR = lm_root_candidate_dir
    lm_root = next(ASL_SIGNS_DIR.rglob('train_landmark_files'), ASL_SIGNS_DIR)
else:
    ASL_SIGNS_DIR = lm_root_candidate.parent
    lm_root       = lm_root_candidate
print(f'ASL Signs   : {ASL_SIGNS_DIR}')
print(f'LM root     : {lm_root}')

# ── Resolve ASL Fingerspelling dataset (optional) ─────────────────────
# Unique to asl-fingerspelling: train.csv with a 'phrase' column
ASL_FS_DIR = None
for csv_path in KAGGLE_INPUT.rglob('train.csv'):
    if csv_path.parent == ASL_SIGNS_DIR: continue   # skip the signs CSV
    try:
        cols = pd.read_csv(csv_path, nrows=1).columns.tolist()
        if 'phrase' in cols:
            ASL_FS_DIR = csv_path.parent
            break
    except Exception:
        continue
HAS_FS = ASL_FS_DIR is not None
print(f'ASL Fingersp: {ASL_FS_DIR if HAS_FS else "NOT ATTACHED — Section 2 will be skipped"}')

# ── Load vocabularies ─────────────────────────────────────────────────
with open(next(PROJECT_DIR.rglob('sign_to_idx.json'))) as f:
    sign_to_idx_full = json.load(f)   # may have 253 if updated

# Load model first to find how many sign classes it actually supports,
# then restrict sign_to_idx to only the model's output classes.
# This handles the 250/253 mismatch between v2 weights and updated vocab.
from model import load_model
model = load_model(str(weights), DEVICE)
model.eval()
MODEL_NUM_SIGNS = model.cfg['num_signs']
print(f'Model sign classes: {MODEL_NUM_SIGNS}')

# Filter sign_to_idx to only indices the model knows
sign_to_idx = {k: v for k, v in sign_to_idx_full.items() if v < MODEL_NUM_SIGNS}
idx_to_sign = {v: k for k, v in sign_to_idx.items()}
NUM_SIGNS   = len(sign_to_idx)

if NUM_SIGNS < len(sign_to_idx_full):
    skipped = sorted(k for k,v in sign_to_idx_full.items() if v >= MODEL_NUM_SIGNS)
    print(f'NOTE: {len(skipped)} signs in sign_to_idx.json exceed model capacity '
          f'and are excluded from evaluation: {", ".join(skipped)}')

with open(next(PROJECT_DIR.rglob('vocabulary.json'))) as f:
    vocab = json.load(f)
idx_to_char = {int(k): v for k, v in vocab['idx_to_char'].items()}
BLANK_IDX   = vocab.get('blank_idx', 0)

print(f'Signs evaluated : {NUM_SIGNS}')
print(f'Vocab chars     : {len(idx_to_char)}')
print(f'Parameters      : {sum(p.numel() for p in model.parameters()):,}')
print('\nAll files found.')

Device: cuda

=== Attached datasets ===
  [competitions]  94603 files
  [datasets]  9 files

PROJECT_DIR : /kaggle/input/datasets/cybertechsavvy/asl-project
Weights     : asl_model_weights_v2.pt  (14.2 MB)
ASL Signs   : /kaggle/input/competitions/asl-signs
LM root     : /kaggle/input/competitions/asl-signs/train_landmark_files
ASL Fingersp: /kaggle/input/competitions/asl-fingerspelling
Model sign classes: 250
NOTE: 3 signs in sign_to_idx.json exceed model capacity and are excluded from evaluation: iloveyou, me, you
Signs evaluated : 250
Vocab chars     : 60
Parameters      : 3,551,030

All files found.


In [2]:
# ── Cell 2: Define normalization and sequence loading helpers ─────────
# Model is already loaded in Cell 1.
import pyarrow.parquet as pq_lib

NUM_FACE, NUM_POSE, NUM_HAND = 76, 12, 21

# Our 130-landmark index sets (from extract_landmarks.py)
_LIPS_O   = [61,146,91,181,84,17,314,405,321,375,291,185,40,39,37,0,267,269,270,409]
_LIPS_I   = [78,95,88,178,87,14,317,402,318,324,308,191,80,81,82,13,312,311,310,415]
_NOSE     = [1,4]
_L_EYE    = [33,160,158,133,153,144,145,154]
_R_EYE    = [362,385,387,263,373,380,374,381]
_L_BROW   = [70,63,105,66,107,55,65,52,53]
_R_BROW   = [336,296,334,293,300,285,295,282,283]
FACE_IDXS = _LIPS_O+_LIPS_I+_NOSE+_L_EYE+_R_EYE+_L_BROW+_R_BROW  # 76 indices
POSE_IDXS = [11,12,13,14,15,16,17,18,19,20,21,22]                   # 12 indices
# Hand indices: 0-20 for both left and right
LHAND_IDXS = list(range(21))
RHAND_IDXS = list(range(21))

# Pre-build lookup: (type_str, landmark_index) → position in our 130 array
# Layout: face[0:76], pose[76:88], left_hand[88:109], right_hand[109:130]
_LM_POS = {}  # (type, lm_idx) -> position 0..129
for pos, face_i  in enumerate(FACE_IDXS):  _LM_POS[('face',       face_i)] = pos
for pos, pose_i  in enumerate(POSE_IDXS):  _LM_POS[('pose',       pose_i)] = 76  + pos
for pos, lh_i    in enumerate(LHAND_IDXS): _LM_POS[('left_hand',  lh_i)]   = 88  + pos
for pos, rh_i    in enumerate(RHAND_IDXS): _LM_POS[('right_hand', rh_i)]   = 109 + pos

def normalize_landmarks(seq):
    seq = seq.copy().astype(np.float32)
    LH, RH = NUM_FACE+NUM_POSE, NUM_FACE+NUM_POSE+NUM_HAND
    for hs in (LH, RH):
        h = seq[:,hs:hs+NUM_HAND,:]; hc = h - h[:,0:1,:]
        sc = np.nanmax(np.linalg.norm(hc,axis=-1,keepdims=True),axis=1,keepdims=True)+1e-6
        seq[:,hs:hs+NUM_HAND,:] = hc/sc
    p = seq[:,NUM_FACE:NUM_FACE+NUM_POSE,:]
    mid = (p[:,0:1,:]+p[:,1:2,:])/2
    sw  = np.linalg.norm(p[:,0:1,:]-p[:,1:2,:],axis=-1,keepdims=True)+1e-6
    seq[:,NUM_FACE:NUM_FACE+NUM_POSE,:] = (p-mid)/sw
    f = seq[:,:NUM_FACE,:]; nose = f[:,40:41,:]
    lec = f[:,42:50,:].mean(axis=1,keepdims=True)
    rec = f[:,50:58,:].mean(axis=1,keepdims=True)
    ed  = np.linalg.norm(lec-rec,axis=-1,keepdims=True)+1e-6
    seq[:,:NUM_FACE,:] = (f-nose)/ed
    return np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0)

def greedy_ctc_decode(log_probs):
    tokens = log_probs.argmax(axis=-1).tolist()
    decoded, prev = [], None
    for t in tokens:
        if t != BLANK_IDX and t != prev:
            decoded.append(idx_to_char.get(t, '?'))
        prev = t
    return ''.join(decoded).upper()

def load_sequence(pq_path, max_len=128):
    """
    Load one asl-signs parquet (long format) → (padded, actual_len, raw_arr).

    The asl-signs parquet is in long format:
      columns: frame, row_id, type, landmark_index, x, y, z
      one row per landmark per frame  (T frames × 543 landmarks = T×543 rows)

    We pivot to wide (T, 130, 3) keeping only our 130 landmarks.

    Returns:
      padded     : (max_len, 130, 3) float32, normalized
      actual_len : int ≥ 1
      raw_arr    : (T, 130, 3) float32, unnormalized
    """
    df = pd.read_parquet(pq_path, columns=['frame','type','landmark_index','x','y','z'])

    if len(df) == 0:
        raise ValueError(f'Empty parquet: {pq_path}')

    # Filter to landmark types we care about
    type_mask = df['type'].isin(('face','pose','left_hand','right_hand'))
    df = df[type_mask].reset_index(drop=True)

    # Map frame values to contiguous 0..T-1 indices
    frame_vals  = df['frame'].to_numpy()
    unique_frames, frame_inv = np.unique(frame_vals, return_inverse=True)
    T = len(unique_frames)
    if T == 0:
        raise ValueError(f'No usable frames: {pq_path}')

    # Map (type, landmark_index) pairs to our 0..129 position using vectorised ops
    # Build integer keys: type_id * 1000 + landmark_index
    type_to_id = {'face': 0, 'pose': 1, 'left_hand': 2, 'right_hand': 3}
    type_ids   = df['type'].map(type_to_id).to_numpy(dtype=np.int32)
    lm_indices = df['landmark_index'].to_numpy(dtype=np.int32)
    int_keys   = type_ids * 10000 + lm_indices

    # Build lookup array: int_key → lm_position (or -1 if not in our 130)
    max_key    = int_keys.max() + 1
    key_to_pos = np.full(max_key, -1, dtype=np.int32)
    for (t_str, lm_i), pos in _LM_POS.items():
        k = type_to_id[t_str] * 10000 + lm_i
        if k < max_key:
            key_to_pos[k] = pos
    lm_positions = key_to_pos[int_keys]   # -1 for unwanted landmarks

    # Keep only rows for our 130 landmarks
    valid = lm_positions >= 0
    fi    = frame_inv[valid]
    pos   = lm_positions[valid]
    xs    = df['x'].to_numpy(dtype=np.float32)[valid]
    ys    = df['y'].to_numpy(dtype=np.float32)[valid]
    zs    = df['z'].to_numpy(dtype=np.float32)[valid]

    # Scatter into (T, 130, 3) array
    arr = np.zeros((T, 130, 3), dtype=np.float32)
    np.nan_to_num(xs, copy=False, nan=0.0)
    np.nan_to_num(ys, copy=False, nan=0.0)
    np.nan_to_num(zs, copy=False, nan=0.0)
    arr[fi, pos, 0] = xs
    arr[fi, pos, 1] = ys
    arr[fi, pos, 2] = zs

    seq = normalize_landmarks(arr)
    al  = max(1, min(T, max_len))
    pad = np.zeros((max_len, 130, 3), dtype=np.float32)
    pad[:al] = seq[:al]
    return pad, al, arr

print('Helpers defined.')
print(f'Landmark lookup table: {len(_LM_POS)} entries (130 landmarks × 4 types)')

Helpers defined.
Landmark lookup table: 130 entries (130 landmarks × 4 types)


In [3]:
# ── Cell 3: Build validation split from ASL Signs ─────────────────────
# ASL_SIGNS_DIR and lm_root are resolved in Cell 1.

# Find the train.csv that has a 'sign' column
signs_csv_path = next(
    (p for p in ASL_SIGNS_DIR.rglob('train.csv')
     if 'sign' in pd.read_csv(p, nrows=1).columns.tolist()),
    None
)
if signs_csv_path is None:
    # Try the direct path
    signs_csv_path = ASL_SIGNS_DIR / 'train.csv'

train_df   = pd.read_csv(signs_csv_path)
print(f'train.csv columns: {list(train_df.columns)}')
print(f'train.csv rows   : {len(train_df)}')

# Only keep signs our model knows (respecting MODEL_NUM_SIGNS limit)
known_mask = train_df['sign'].isin(sign_to_idx)
df         = train_df[known_mask].copy().reset_index(drop=True)

# Stratified 20% val split by participant — tests new-signer generalisation
val_rows = []
for sign, grp in df.groupby('sign'):
    parts = sorted(grp['participant_id'].unique())
    n_val = max(1, int(len(parts) * 0.20))
    vp    = set(parts[-n_val:])
    val_rows.append(grp[grp['participant_id'].isin(vp)])
val_df = pd.concat(val_rows).reset_index(drop=True)

print(f'\nTotal samples (model-known signs) : {len(df)}')
print(f'Validation samples                : {len(val_df)}')
print(f'Signs covered                     : {val_df["sign"].nunique()} / {NUM_SIGNS}')
vc = val_df['sign'].value_counts()
print(f'Samples per sign                  : min={vc.min()} max={vc.max()} mean={vc.mean():.1f}')

train.csv columns: ['path', 'participant_id', 'sequence_id', 'sign']
train.csv rows   : 94477

Total samples (model-known signs) : 94477
Validation samples                : 18906
Signs covered                     : 250 / 250
Samples per sign                  : min=62 max=95 mean=75.6


In [4]:
# ── Cell 4: SECTION 1 — Signs head batch evaluation ───────────────────
from collections import defaultdict

# ── Smoke test: verify parquet path resolution before running anything ─
# The 'path' column in train.csv may be relative to ASL_SIGNS_DIR or to
# lm_root — we test both and pick whichever resolves to a real file.
sample_row = val_df.iloc[0]
raw_path   = sample_row['path']   # e.g. '12345/678901234.parquet'

candidate_a = lm_root       / raw_path   # train_landmark_files/<path>
candidate_b = ASL_SIGNS_DIR / raw_path   # dataset_root/<path>

print('=== Smoke test ===')
print(f'raw path value : {raw_path}')
print(f'candidate A    : {candidate_a}  exists={candidate_a.exists()}')
print(f'candidate B    : {candidate_b}  exists={candidate_b.exists()}')

if candidate_a.exists():
    PQ_ROOT = lm_root
    print('Using candidate A (lm_root / path)')
elif candidate_b.exists():
    PQ_ROOT = ASL_SIGNS_DIR
    print('Using candidate B (ASL_SIGNS_DIR / path)')
else:
    # Last resort: search for the file by name
    fname   = Path(raw_path).name
    matches = list(lm_root.rglob(fname)) or list(ASL_SIGNS_DIR.rglob(fname))
    if matches:
        PQ_ROOT = matches[0].parent.parent   # go up past participant dir
        print(f'Found via rglob: {matches[0]}')
        print(f'PQ_ROOT set to : {PQ_ROOT}')
    else:
        # Print first 5 actual parquet paths to show real structure
        actual = list(lm_root.rglob('*.parquet'))[:5]
        raise FileNotFoundError(
            f'Cannot resolve parquet path.\n'
            f'raw_path = {raw_path}\n'
            f'lm_root  = {lm_root}\n'
            f'Sample actual parquets: {[str(p) for p in actual]}'
        )

# Quick load test
try:
    pad_test, al_test, _ = load_sequence(PQ_ROOT / raw_path)
    print(f'Test load OK: shape={pad_test.shape}  actual_len={al_test}')
except Exception as e:
    raise RuntimeError(f'load_sequence failed on sample file: {e}')
print('===================\n')

# ── Batch inference ───────────────────────────────────────────────────
BATCH = 32

all_true_list = []
all_top5_list = []   # each element: (5,) int array
all_conf_list = []   # each element: (NUM_SIGNS,) float array
load_errors   = []

buf_seqs, buf_lens, buf_labels = [], [], []

@torch.no_grad()
def flush_batch():
    x      = torch.from_numpy(np.stack(buf_seqs)).to(DEVICE)
    s_len  = torch.tensor(buf_lens, dtype=torch.long).to(DEVICE)
    logits = model(x, s_len, mode='signs')
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()         # (B, num_signs)
    t5_i   = np.argsort(probs, axis=-1)[:, ::-1][:, :5].copy()  # (B, 5)
    for i in range(len(buf_labels)):
        all_true_list.append(buf_labels[i])
        all_top5_list.append(t5_i[i])        # (5,) int array
        all_conf_list.append(probs[i])        # (num_signs,) float array
    buf_seqs.clear(); buf_lens.clear(); buf_labels.clear()

t0 = time.time()
for i_row, (_, row) in enumerate(val_df.iterrows()):
    pq = PQ_ROOT / row['path']
    if not pq.exists():
        load_errors.append(str(row['path']))
        continue
    try:
        pad, al, _ = load_sequence(pq)
    except Exception as e:
        load_errors.append(f"{row['path']}: {e}")
        continue
    buf_seqs.append(pad)
    buf_lens.append(al)
    buf_labels.append(sign_to_idx[row['sign']])
    if len(buf_seqs) == BATCH:
        flush_batch()
    if (i_row + 1) % 2000 == 0:
        print(f'  processed {i_row+1}/{len(val_df)}  '
              f'({len(all_true_list)} loaded, {len(load_errors)} errors)  '
              f'{time.time()-t0:.0f}s')
if buf_seqs:
    flush_batch()

# ── Defensive stacking with shape assertions ──────────────────────────
if len(all_true_list) == 0:
    raise RuntimeError(
        f'No samples were loaded successfully.\n'
        f'Load errors ({len(load_errors)}): {load_errors[:5]}\n'
        f'PQ_ROOT: {PQ_ROOT}'
    )

all_true = np.array(all_true_list, dtype=np.int64)               # (N,)
all_top5 = np.stack(all_top5_list, axis=0).astype(np.int64)      # (N, 5)
all_conf = np.stack(all_conf_list, axis=0).astype(np.float32)    # (N, num_signs)
N = len(all_true)

assert all_top5.ndim == 2 and all_top5.shape[1] == 5, \
    f'all_top5 shape wrong: {all_top5.shape}'
assert all_conf.ndim == 2, \
    f'all_conf shape wrong: {all_conf.shape}'

# ── Overall top-k accuracy ────────────────────────────────────────────
top1 = float((all_top5[:, 0] == all_true).mean())
top3 = float(np.any(all_top5[:, :3] == all_true[:, None], axis=1).mean())
top5 = float(np.any(all_top5[:, :5] == all_true[:, None], axis=1).mean())

print(f'[Signs head — {N} samples  ({len(load_errors)} failed to load)]')
print(f'  Top-1 : {top1*100:.2f}%')
print(f'  Top-3 : {top3*100:.2f}%')
print(f'  Top-5 : {top5*100:.2f}%')
print(f'  Chance: {100/NUM_SIGNS:.2f}%')
print(f'  Time  : {time.time()-t0:.1f}s')
if load_errors:
    print(f'  First 3 load errors: {load_errors[:3]}')

# ── Per-class accuracy ────────────────────────────────────────────────
per_class = {}
for sign, idx in sign_to_idx.items():
    mask = all_true == idx
    n    = int(mask.sum())
    if n == 0:
        per_class[sign] = dict(top1=0.0, top3=0.0, top5=0.0, n=0,
                               conf_correct=0.0, conf_wrong=0.0)
        continue
    t1 = float((all_top5[mask, 0] == idx).mean())
    t3 = float(np.any(all_top5[mask, :3] == idx, axis=1).mean())
    t5 = float(np.any(all_top5[mask, :5] == idx, axis=1).mean())
    cok = mask & (all_top5[:, 0] == idx)
    cwg = mask & (all_top5[:, 0] != idx)
    per_class[sign] = dict(
        top1=t1, top3=t3, top5=t5, n=n,
        conf_correct=float(all_conf[cok, idx].mean()) if cok.any() else 0.0,
        conf_wrong  =float(all_conf[cwg, all_top5[cwg, 0]].mean()) if cwg.any() else 0.0,
    )

# ── Confusion pairs ───────────────────────────────────────────────────
wrong = all_top5[:, 0] != all_true
pair_counts = defaultdict(int)
for t, p in zip(all_true[wrong], all_top5[wrong, 0]):
    pair_counts[(idx_to_sign.get(int(t), str(t)),
                 idx_to_sign.get(int(p), str(p)))] += 1
top_confusions = sorted(pair_counts.items(), key=lambda x: -x[1])[:30]

# ── Confidence calibration ────────────────────────────────────────────
top_conf = all_conf[np.arange(N), all_top5[:, 0]]
correct  = (all_top5[:, 0] == all_true).astype(float)
bins     = np.linspace(0, 1, 11)
cal_data = []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (top_conf >= lo) & (top_conf < hi)
    if m.sum() > 0:
        cal_data.append((f'{lo:.1f}–{hi:.1f}',
                         float(correct[m].mean()),
                         float(top_conf[m].mean()),
                         int(m.sum())))

print('\nCalibration (accuracy vs avg confidence in each bucket):')
for lbl, acc, conf, cnt in cal_data:
    bar = '█' * int(acc * 20)
    print(f'  {lbl}: acc={acc*100:5.1f}%  conf={conf*100:5.1f}%  n={cnt:5d}  {bar}')

sorted_pc = sorted(per_class.items(), key=lambda x: x[1]['top1'])
print('\nBottom 10 signs:')
for s, m in sorted_pc[:10]:
    print(f'  {s:<22} top1={m["top1"]*100:5.1f}%  n={m["n"]}')
print('\nTop 10 signs:')
for s, m in sorted_pc[-10:]:
    print(f'  {s:<22} top1={m["top1"]*100:5.1f}%  n={m["n"]}')

=== Smoke test ===
raw path value : train_landmark_files/61333/1002052130.parquet
candidate A    : /kaggle/input/competitions/asl-signs/train_landmark_files/train_landmark_files/61333/1002052130.parquet  exists=False
candidate B    : /kaggle/input/competitions/asl-signs/train_landmark_files/61333/1002052130.parquet  exists=True
Using candidate B (ASL_SIGNS_DIR / path)
Test load OK: shape=(128, 130, 3)  actual_len=116

  processed 2000/18906  (1984 loaded, 0 errors)  43s
  processed 4000/18906  (4000 loaded, 0 errors)  85s
  processed 6000/18906  (5984 loaded, 0 errors)  126s
  processed 8000/18906  (8000 loaded, 0 errors)  166s
  processed 10000/18906  (9984 loaded, 0 errors)  206s
  processed 12000/18906  (12000 loaded, 0 errors)  247s
  processed 14000/18906  (13984 loaded, 0 errors)  289s
  processed 16000/18906  (16000 loaded, 0 errors)  331s
  processed 18000/18906  (17984 loaded, 0 errors)  376s
[Signs head — 18906 samples  (0 failed to load)]
  Top-1 : 72.40%
  Top-3 : 87.90%
  

In [5]:
# ── Cell 5: SECTION 2 — Fingerspelling head CER evaluation ────────────
#
# Character Error Rate (CER) = edit_distance(decoded, target) / len(target)
# Lower is better.  0% = perfect.  100% = completely wrong.
#
# The asl-fingerspelling dataset has parquet files with the same MediaPipe
# landmark format as asl-signs, plus a 'phrase' column with the ground-truth
# character sequence.

if not HAS_FS:
    print('ASL Fingerspelling dataset not attached — skipping Section 2.')
    print('To run this section: Add data → Competitions → '
          'Google ASL Fingerspelling Recognition')
    cer_results = None
else:
    import pandas as pd

    def edit_distance(a, b):
        """Standard Levenshtein edit distance."""
        m, n = len(a), len(b)
        dp = list(range(n+1))
        for i in range(1, m+1):
            prev, dp[0] = dp[0], i
            for j in range(1, n+1):
                prev, dp[j] = dp[j], (prev if a[i-1]==b[j-1]
                                      else 1+min(prev, dp[j], dp[j-1]))
        return dp[n]

    fs_csv_path = next(ASL_FS_DIR.rglob('train.csv'), None)
    if fs_csv_path is None:
        print('train.csv not found in fingerspelling dataset.')
        cer_results = None
    else:
        fs_df   = pd.read_csv(fs_csv_path).dropna(subset=['phrase'])
        fs_root = fs_csv_path.parent / 'train_landmarks'
        if not fs_root.exists():
            fs_root = next((p.parent for p in fs_csv_path.parent.rglob('*.parquet')),
                           fs_csv_path.parent)

        # Sample up to 500 clips for speed
        sample_df = fs_df.sample(min(500, len(fs_df)), random_state=42)

        cer_list, decoded_samples = [], []
        t0 = time.time()

        for _, row in sample_df.iterrows():
            pq = fs_root / f"{row.get('file_id', row.get('sequence_id', ''))}.parquet"
            if not pq.exists(): continue
            try:
                pad, al, _ = load_sequence(pq)
            except Exception:
                continue

            with torch.no_grad():
                x     = torch.from_numpy(pad).unsqueeze(0).to(DEVICE)
                s_len = torch.tensor([al], dtype=torch.long).to(DEVICE)
                lp    = model(x, s_len, mode='fingerspelling')
            decoded = greedy_ctc_decode(lp[0].cpu().numpy())
            target  = str(row['phrase']).upper()
            cer     = edit_distance(decoded, target) / max(len(target), 1)
            cer_list.append(cer)
            if len(decoded_samples) < 20:
                decoded_samples.append((target, decoded, cer))

        mean_cer = np.mean(cer_list) if cer_list else 1.0
        cer_results = dict(mean_cer=mean_cer, samples=decoded_samples,
                           n=len(cer_list))
        print(f'[Fingerspelling head — {len(cer_list)} samples]')
        print(f'  Mean CER : {mean_cer*100:.1f}%  (lower = better, 0% = perfect)')
        print(f'  Accuracy : {(1-mean_cer)*100:.1f}%')
        print(f'  Elapsed  : {time.time()-t0:.1f}s')
        print('\n  Sample decoded outputs (target → decoded):')
        for tgt, dec, cer in decoded_samples[:8]:
            print(f'    {tgt:<20} → {dec:<20}  CER={cer*100:.0f}%')

[Fingerspelling head — 0 samples]
  Mean CER : 100.0%  (lower = better, 0% = perfect)
  Accuracy : 0.0%
  Elapsed  : 12.7s

  Sample decoded outputs (target → decoded):


In [6]:
# ── Cell 6: SECTION 3 — Continuous feed simulation ────────────────────
#
# This is the most realistic validation.
#
# For each validation clip we simulate the EXACT webcam loop:
#   1. Start with an empty deque(maxlen=64) — the sliding buffer
#   2. Feed the raw landmark frames one at a time, exactly as the
#      webcam extractor would
#   3. Call predict() after each frame (once buffer has >= MIN_FRAMES)
#   4. Run format_token() and update() on CaptionState — exactly as
#      run_captions.py does
#   5. Record whether the correct sign is eventually committed,
#      how many frames it took, and whether any wrong sign was committed
#
# This tests the full pipeline: model + confidence threshold +
# hold-frame deduplication + commit logic — not just the raw model output.

import collections, time
from typing import Optional

# ── Pipeline constants (must match run_captions.py exactly) ───────────
CONFIDENCE_THRESH  = 0.35
COMMIT_HOLD_FRAMES = 8
BUFFER_SIZE        = 64
MIN_FRAMES         = 10

# ── Inline CaptionState (exact copy from run_captions.py) ─────────────
class CaptionState:
    def __init__(self):
        self.committed       = []
        self.live_token      = None
        self._hold_count     = 0
        self._last_committed = None
    def update(self, token):
        if token is None:
            self.live_token = None; self._hold_count = 0; return
        self.live_token = token
        if token == self._last_committed: self._hold_count = 0; return
        if self.committed and self.committed[-1] == token:
            self._hold_count = 0; return
        if self.live_token == token: self._hold_count += 1
        else: self._hold_count = 1
        if self._hold_count >= COMMIT_HOLD_FRAMES:
            self.committed.append(token)
            self._last_committed = token; self._hold_count = 0
    @property
    def hold_progress(self):
        return min(self._hold_count / COMMIT_HOLD_FRAMES, 1.0)

def format_token(result):
    if not result['ready']: return None
    fs = result.get('fs','').strip().upper()
    if fs: return f'fs-{fs}'
    sign = result.get('sign','')
    conf = result.get('conf', 0.0)
    if sign and conf >= CONFIDENCE_THRESH: return sign.upper()
    return None

@torch.no_grad()
def predict_from_buffer(buf, actual_len):
    """Run both heads on the current deque contents."""
    seq = np.stack(list(buf), axis=0)     # (T, 130, 3)
    seq = normalize_landmarks(seq)
    T   = seq.shape[0]
    al  = min(T, 128)
    pad = np.zeros((128, 130, 3), dtype=np.float32)
    pad[:al] = seq[:al]
    x     = torch.from_numpy(pad).unsqueeze(0).to(DEVICE)
    s_len = torch.tensor([al], dtype=torch.long).to(DEVICE)
    lp_fs = model(x, s_len, mode='fingerspelling')
    lg_sg = model(x, s_len, mode='signs')
    fs    = greedy_ctc_decode(lp_fs[0].cpu().numpy())
    prob  = torch.softmax(lg_sg[0], dim=-1).cpu().numpy()
    top1  = int(prob.argmax())
    return {'ready': True,
            'fs'   : fs,
            'sign' : idx_to_sign.get(top1, ''),
            'conf' : float(prob[top1])}

# ── Run simulation on a sample of the val set ─────────────────────────
# Cap at 400 clips for speed (~2–5 min on GPU)
SIM_SAMPLE  = 400
sim_df = val_df.sample(min(SIM_SAMPLE, len(val_df)), random_state=42)

sim_results = []   # one dict per clip
t0 = time.time()

for _, row in sim_df.iterrows():
    pq = PQ_ROOT / row['path']
    if not pq.exists(): continue
    try:
        _, _, raw_seq = load_sequence(pq)   # raw (T, 130, 3) unnormalized
    except Exception:
        continue

    true_sign = row['sign'].upper()
    buf       = collections.deque(maxlen=BUFFER_SIZE)
    caption   = CaptionState()
    T         = raw_seq.shape[0]

    committed_correct  = False
    committed_wrong    = False
    commit_frame       = None   # frame index when correct sign was committed
    first_correct_live = None   # frame index when correct sign first appeared as live token

    for frame_i, frame in enumerate(raw_seq):
        buf.append(frame)
        if len(buf) < MIN_FRAMES:
            continue

        result = predict_from_buffer(buf, len(buf))
        token  = format_token(result)
        caption.update(token)

        # Track first time correct sign appears as live token
        if (first_correct_live is None and
                caption.live_token is not None and
                caption.live_token.upper() == true_sign):
            first_correct_live = frame_i

        # Check committed list after each frame
        if true_sign in [c.upper() for c in caption.committed]:
            committed_correct = True
            if commit_frame is None:
                commit_frame = frame_i
        if caption.committed and true_sign not in [c.upper() for c in caption.committed]:
            committed_wrong = True

    sim_results.append({
        'sign'             : true_sign,
        'n_frames'         : T,
        'committed_correct': committed_correct,
        'committed_wrong'  : committed_wrong,
        'commit_frame'     : commit_frame,
        'first_live_frame' : first_correct_live,
        'final_committed'  : list(caption.committed),
    })

# ── Aggregate metrics ─────────────────────────────────────────────────
n_sim         = len(sim_results)
n_correct     = sum(r['committed_correct'] for r in sim_results)
n_wrong       = sum(r['committed_wrong']   for r in sim_results)
n_silent      = sum(not r['committed_correct'] and not r['committed_wrong']
                    for r in sim_results)
commit_frames = [r['commit_frame'] for r in sim_results if r['commit_frame'] is not None]
live_frames   = [r['first_live_frame'] for r in sim_results if r['first_live_frame'] is not None]

pipeline_acc   = n_correct / n_sim
false_pos_rate = n_wrong   / n_sim
silence_rate   = n_silent  / n_sim
avg_commit_f   = np.mean(commit_frames)  if commit_frames else float('nan')
avg_live_f     = np.mean(live_frames)    if live_frames   else float('nan')

print(f'[Continuous feed simulation — {n_sim} clips]')
print(f'  Correct sign committed  : {pipeline_acc*100:.1f}%  ({n_correct}/{n_sim})')
print(f'  Wrong sign committed    : {false_pos_rate*100:.1f}%  ({n_wrong}/{n_sim})')
print(f'  No sign committed       : {silence_rate*100:.1f}%  ({n_silent}/{n_sim})')
print(f'  Avg frames to commit    : {avg_commit_f:.1f}  (of avg {np.mean([r["n_frames"] for r in sim_results]):.1f} total)')
print(f'  Avg frames to live view : {avg_live_f:.1f}')
print(f'  Elapsed : {time.time()-t0:.1f}s')

[Continuous feed simulation — 400 clips]
  Correct sign committed  : 25.5%  (102/400)
  Wrong sign committed    : 51.0%  (204/400)
  No sign committed       : 31.8%  (127/400)
  Avg frames to commit    : 28.5  (of avg 36.4 total)
  Avg frames to live view : 21.0
  Elapsed : 80.2s


In [7]:
# ── Cell 7: SECTION 4 — Confidence threshold sweep ────────────────────
#
# The CONFIDENCE_THRESH in run_captions.py controls the tradeoff between:
#   - Precision : only commit when very sure  (higher threshold)
#   - Recall    : catch more signs            (lower threshold)
#
# We sweep over candidate values and show the resulting pipeline accuracy
# and false positive rate using the batch signs-head scores (fast proxy
# for the continuous simulation).

thresholds    = [0.10, 0.20, 0.30, 0.35, 0.40, 0.50, 0.60, 0.70, 0.80]
thresh_results = []

top1_conf     = all_conf[np.arange(N), all_top5[:,0]]
is_correct    = (all_top5[:,0] == all_true)

for thresh in thresholds:
    fires       = top1_conf >= thresh            # model would emit a token
    true_pos    = (fires & is_correct).sum()     # correct fires
    false_pos   = (fires & ~is_correct).sum()    # wrong fires
    missed      = (~fires & is_correct).sum()    # below threshold, skipped
    n_fires     = fires.sum()
    precision   = true_pos / max(n_fires, 1)
    recall      = true_pos / max(is_correct.sum(), 1)
    thresh_results.append(dict(
        threshold=thresh, precision=precision, recall=recall,
        false_pos=int(false_pos), missed=int(missed), n_fires=int(n_fires)
    ))
    marker = ' ← current' if abs(thresh - 0.35) < 0.01 else ''
    print(f'  thresh={thresh:.2f}  precision={precision*100:5.1f}%  '
          f'recall={recall*100:5.1f}%  false_pos={false_pos}  missed={missed}{marker}')

print('\nNote: precision = fraction of committed signs that are correct.')
print('      recall    = fraction of all correct signs that get committed.')

  thresh=0.10  precision= 73.0%  recall= 99.9%  false_pos=5047  missed=16
  thresh=0.20  precision= 75.2%  recall= 99.0%  false_pos=4466  missed=141
  thresh=0.30  precision= 78.4%  recall= 96.6%  false_pos=3642  missed=464
  thresh=0.35  precision= 80.1%  recall= 94.8%  false_pos=3230  missed=709 ← current
  thresh=0.40  precision= 81.8%  recall= 92.8%  false_pos=2825  missed=984
  thresh=0.50  precision= 85.8%  recall= 86.9%  false_pos=1969  missed=1787
  thresh=0.60  precision= 89.6%  recall= 79.3%  false_pos=1256  missed=2836
  thresh=0.70  precision= 93.0%  recall= 71.0%  false_pos=730  missed=3967
  thresh=0.80  precision= 95.9%  recall= 61.0%  false_pos=355  missed=5340

Note: precision = fraction of committed signs that are correct.
      recall    = fraction of all correct signs that get committed.


In [8]:
# ── Cell 8: SECTION 5 — Silence rejection test ────────────────────────
#
# Feeds the model clips of the SAME sign repeated back-to-back with a
# gap of zero-landmark frames between them (simulating the signer
# dropping their hands between signs).
# Measures whether CaptionState correctly suppresses the duplicate
# and whether the inter-sign gap produces false positives.
#
# Also runs a pure silence test: feeds only zero-landmark frames
# (hands at sides, no signing) and counts any false commits.

SILENCE_FRAMES = 30   # frames of silence between signs
silence_frame  = np.zeros((130, 3), dtype=np.float32)

# ── Test A: Pure silence ──────────────────────────────────────────────
buf     = collections.deque(maxlen=BUFFER_SIZE)
caption = CaptionState()
false_fires_silence = 0

for _ in range(128):   # 128 silence frames
    buf.append(silence_frame)
    if len(buf) < MIN_FRAMES: continue
    result = predict_from_buffer(buf, len(buf))
    token  = format_token(result)
    caption.update(token)
false_fires_silence = len(caption.committed)
print(f'Pure silence (128 frames): {false_fires_silence} false commits')
if caption.committed:
    print(f'  False commits: {caption.committed}')

# ── Test B: Sign → silence → same sign (duplicate suppression) ────────
# Pick 5 random signs with sufficient val samples
test_signs = [s for s, m in per_class.items() if m['n'] >= 3]
np.random.seed(42)
test_signs = list(np.random.choice(test_signs, size=min(5, len(test_signs)), replace=False))

dup_results = []
for test_sign in test_signs:
    # Get one clip for this sign
    clip_row = val_df[val_df['sign'] == test_sign].iloc[0]
    pq = PQ_ROOT / clip_row['path']
    if not pq.exists(): continue
    try:
        _, _, raw = load_sequence(pq)
    except Exception:
        continue

    # Feed: sign1 → silence → sign1 again
    buf     = collections.deque(maxlen=BUFFER_SIZE)
    caption = CaptionState()

    sequence = list(raw) + [silence_frame]*SILENCE_FRAMES + list(raw)
    for frame in sequence:
        buf.append(frame)
        if len(buf) < MIN_FRAMES: continue
        result = predict_from_buffer(buf, len(buf))
        token  = format_token(result)
        caption.update(token)

    correct_count = sum(1 for c in caption.committed if c.upper() == test_sign.upper())
    dup_results.append({
        'sign': test_sign, 'committed': caption.committed,
        'correct_count': correct_count,
        'duplicates': max(0, correct_count - 2)   # expect at most 2 (one per sign)
    })

print('\nDuplicate suppression test (sign → pause → same sign):')
print(f'  Expected: each sign committed exactly 1–2 times')
for r in dup_results:
    status = 'OK' if r['correct_count'] <= 2 else 'DUPLICATE'
    print(f'  {r["sign"]:<20} committed={r["committed"]}  [{status}]')

Pure silence (128 frames): 0 false commits

Duplicate suppression test (sign → pause → same sign):
  Expected: each sign committed exactly 1–2 times
  milk                 committed=['MILK', 'fs-/', 'fs-//', 'fs-1', 'fs-J', 'MILK', 'fs-X', 'fs-11  /']  [OK]
  another              committed=['fs-10', 'ANOTHER', 'fs-1']  [OK]
  go                   committed=['fs-1']  [OK]
  donkey               committed=['fs-A', 'fs-AAA', 'fs-AA', 'fs-AAA', 'fs-AAACA']  [OK]
  helicopter           committed=['fs-33', 'fs-3', 'fs-33', 'fs-3 E', 'fs-33', 'fs-323-3', 'fs-3-3']  [OK]


In [9]:
# ── Cell 9: Generate HTML report ─────────────────────────────────────
from pathlib import Path

REPORT = Path('/kaggle/working/validation_report.html')

def pct(v):  return f'{v*100:.1f}%'
def acc_color(v):
    if v >= 0.80: return '#2ecc71'
    if v >= 0.60: return '#a8e6a3'
    if v >= 0.40: return '#f9e79f'
    if v >= 0.20: return '#f0b27a'
    return '#e74c3c'
def badge(v, invert=False):
    """Green badge for high values (or low values if invert=True)."""
    good = v <= 0.3 if invert else v >= 0.7
    col  = '#27ae60' if good else ('#e67e22' if (v<=0.5 if not invert else v>=0.5) else '#e74c3c')
    return f'<span style="background:{col};color:#fff;border-radius:3px;padding:1px 6px;font-size:.82em">{pct(v)}</span>'

# ── Per-class table ────────────────────────────────────────────────────
sorted_pc = sorted(per_class.items(), key=lambda x: x[1]['top1'])
class_rows = ''
for rank, (sign, m) in enumerate(sorted_pc, 1):
    bg = acc_color(m['top1'])
    class_rows += f"""
    <tr class="dr">
      <td>{rank}</td><td><strong>{sign}</strong></td>
      <td style="background:{bg};text-align:center;font-weight:600">{pct(m['top1'])}</td>
      <td style="text-align:center">{pct(m['top3'])}</td>
      <td style="text-align:center">{pct(m['top5'])}</td>
      <td style="text-align:center">{m['n']}</td>
      <td style="text-align:center">{badge(m['conf_correct'])}</td>
      <td style="text-align:center">{badge(m['conf_wrong'], invert=True)}</td>
    </tr>"""

# ── Confusion pairs table ─────────────────────────────────────────────
conf_rows = ''.join(f"""
    <tr><td><strong>{t}</strong></td><td>→</td>
    <td><strong>{p}</strong></td><td style="text-align:center">{c}</td></tr>"""
    for (t,p), c in top_confusions)

# ── Threshold sweep table ─────────────────────────────────────────────
thr_rows = ''.join(f"""
    <tr {'style="background:#eaf4fb;font-weight:600"' if abs(r['threshold']-0.35)<0.01 else ''}>
      <td>{r['threshold']:.2f}{' ← current' if abs(r['threshold']-0.35)<0.01 else ''}</td>
      <td style="text-align:center">{badge(r['precision'])}</td>
      <td style="text-align:center">{badge(r['recall'])}</td>
      <td style="text-align:center">{r['false_pos']}</td>
      <td style="text-align:center">{r['missed']}</td>
      <td style="text-align:center">{r['n_fires']}</td>
    </tr>""" for r in thresh_results)

# ── Calibration SVG ───────────────────────────────────────────────────
cal_svg = ''
for i, (lbl, acc, conf, cnt) in enumerate(cal_data):
    x = 50 + i*68; ha = int(acc*150); hc = int(conf*150)
    cal_svg += f"""
      <rect x="{x}" y="{160-ha}" width="28" height="{ha}" fill="#3498db" opacity=".85"/>
      <rect x="{x+30}" y="{160-hc}" width="28" height="{hc}" fill="#e74c3c" opacity=".6"/>
      <text x="{x+28}" y="175" font-size="8" text-anchor="middle" fill="#777">{lbl}</text>
      <text x="{x+28}" y="186" font-size="7" text-anchor="middle" fill="#aaa">n={cnt}</text>"""

# ── Fingerspelling section ────────────────────────────────────────────
fs_section = ''
if cer_results:
    sample_rows = ''.join(f"""
    <tr><td><code>{t}</code></td><td><code>{d}</code></td>
    <td style="text-align:center">{badge(1-c)}</td></tr>"""
    for t, d, c in cer_results['samples'])
    fs_section = f"""
    <h2>Section 2 — Fingerspelling Head</h2>
    <div class="sec">
      <div class="cards">
        <div class="card"><div class="cv" style="color:{'#27ae60' if cer_results['mean_cer']<0.4 else '#e74c3c'}">{pct(1-cer_results['mean_cer'])}</div><div class="cl">Char Accuracy</div></div>
        <div class="card"><div class="cv" style="color:#e74c3c">{pct(cer_results['mean_cer'])}</div><div class="cl">Mean CER</div></div>
        <div class="card"><div class="cv" style="color:#8e44ad">{cer_results['n']}</div><div class="cl">Samples</div></div>
      </div>
      <p class="desc"><strong>CER (Character Error Rate)</strong> = edit distance between decoded output and ground-truth phrase, divided by phrase length.
      0% = perfect. 100% = completely wrong. The fingerspelling head was not retrained in v2/v3 so this reflects the original training.</p>
      <table style="max-width:600px"><thead><tr><th>Target</th><th>Decoded</th><th>Char Accuracy</th></tr></thead>
      <tbody>{sample_rows}</tbody></table>
    </div>"""
else:
    fs_section = '<h2>Section 2 — Fingerspelling Head</h2><div class="sec"><p class="desc">Skipped — attach the <strong>asl-fingerspelling</strong> competition dataset to run this section.</p></div>'

# ── Sim results ───────────────────────────────────────────────────────
sim_per_sign = defaultdict(lambda: {'correct':0,'wrong':0,'total':0,'frames':[]})
for r in sim_results:
    s = r['sign']
    sim_per_sign[s]['total'] += 1
    if r['committed_correct']: sim_per_sign[s]['correct'] += 1
    if r['committed_wrong']:   sim_per_sign[s]['wrong']   += 1
    if r['commit_frame'] is not None:
        sim_per_sign[s]['frames'].append(r['commit_frame'])

sim_worst = sorted([(s,v) for s,v in sim_per_sign.items() if v['total']>=2],
                    key=lambda x: x[1]['correct']/x[1]['total'])[:20]
sim_rows = ''.join(f"""
    <tr><td><strong>{s}</strong></td>
    <td style="text-align:center;background:{acc_color(v['correct']/v['total'])}">{pct(v['correct']/v['total'])}</td>
    <td style="text-align:center">{v['correct']}/{v['total']}</td>
    <td style="text-align:center">{v['wrong']}</td>
    <td style="text-align:center">{f"{np.mean(v['frames']):.0f}" if v['frames'] else '—'}</td>
    </tr>""" for s,v in sim_worst)

# ── Dup test ──────────────────────────────────────────────────────────
dup_rows = ''.join(f"""
    <tr><td><strong>{r['sign']}</strong></td>
    <td style="text-align:center">{r['correct_count']}</td>
    <td style="text-align:center">{', '.join(r['committed']) or '(none)'}</td>
    <td style="text-align:center;color:{'#27ae60' if r['duplicates']==0 else '#e74c3c'}">
    {'✓ OK' if r['duplicates']==0 else f'⚠ {r["duplicates"]} extra'}</td>
    </tr>""" for r in dup_results)

# ── Priority signs (fails continuous sim) ────────────────────────────
priority_rows = ''.join(f"""
    <tr><td><strong>{s}</strong></td>
    <td style="text-align:center;background:{acc_color(v['correct']/v['total'])}">{pct(v['correct']/v['total'])}</td>
    <td style="text-align:center">{per_class.get(s.lower(),{}).get('top1',0)*100:.1f}%</td>
    <td style="text-align:center">{v['wrong']}</td>
    </tr>""" for s,v in sim_worst if v['correct']/v['total'] < 0.30)

# ── Assemble HTML ─────────────────────────────────────────────────────
html = f"""
<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">
<title>ASL Full Pipeline Validation</title>
<style>
body{{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;margin:0;padding:24px 32px;background:#f7f8fa;color:#222}}
h1{{color:#2c3e50}}h2{{color:#34495e;border-bottom:2px solid #e0e0e0;padding-bottom:6px;margin-top:36px}}
.sub{{color:#888;font-size:.93em;margin-bottom:26px}}
.cards{{display:flex;gap:12px;flex-wrap:wrap;margin:14px 0 24px}}
.card{{background:#fff;border:1px solid #e0e0e0;border-radius:8px;padding:16px 20px;text-align:center;min-width:110px;box-shadow:0 1px 3px rgba(0,0,0,.05)}}
.cv{{font-size:1.8em;font-weight:700}}.cl{{color:#888;font-size:.78em;margin-top:3px}}
.sec{{background:#fff;border-radius:10px;padding:20px 24px;margin-bottom:22px;box-shadow:0 1px 3px rgba(0,0,0,.05)}}
.note{{background:#eaf4fb;border-left:4px solid #3498db;padding:9px 13px;border-radius:0 5px 5px 0;font-size:.86em;color:#2c3e50;margin:12px 0}}
.desc{{color:#666;font-size:.87em;line-height:1.55}}
table{{border-collapse:collapse;width:100%;font-size:.87em}}
th{{background:#2c3e50;color:#fff;padding:8px 11px;text-align:left}}
td{{padding:7px 11px;border-bottom:1px solid #f0f0f0}}
tr.dr:hover td{{background:#f5f7fa}}
input{{padding:5px 9px;border:1px solid #ccc;border-radius:4px;width:200px;font-size:.87em}}
code{{background:#f4f4f4;padding:1px 5px;border-radius:3px;font-size:.9em}}
</style>
<script>
function ft(i,t){{const q=document.getElementById(i).value.toLowerCase();
document.querySelectorAll('#'+t+' tr.dr').forEach(r=>{{
r.style.display=r.textContent.toLowerCase().includes(q)?'':'none';}});}}
</script></head><body>

<h1>ASL Full Pipeline Validation Report</h1>
<div class="sub">Weights: <strong>{weights.name}</strong> &nbsp;·&nbsp;
Signs validation: <strong>{N} samples</strong> &nbsp;·&nbsp;
Pipeline simulation: <strong>{n_sim} clips</strong></div>

<h2>Section 1 — Signs Head (Batch)</h2>
<div class="sec">
  <div class="cards">
    <div class="card"><div class="cv" style="color:{'#27ae60' if top1>.6 else '#e74c3c'}">{pct(top1)}</div><div class="cl">Top-1</div></div>
    <div class="card"><div class="cv" style="color:#3498db">{pct(top3)}</div><div class="cl">Top-3</div></div>
    <div class="card"><div class="cv" style="color:#3498db">{pct(top5)}</div><div class="cl">Top-5</div></div>
    <div class="card"><div class="cv" style="color:#95a5a6">{pct(1/NUM_SIGNS)}</div><div class="cl">Chance</div></div>
    <div class="card"><div class="cv" style="color:#27ae60">{sum(1 for m in per_class.values() if m['n']>0 and m['top1']>=.8)}</div><div class="cl">Signs ≥80%</div></div>
    <div class="card"><div class="cv" style="color:#e74c3c">{sum(1 for m in per_class.values() if m['n']>0 and m['top1']==0)}</div><div class="cl">Signs 0%</div></div>
  </div>
  <div class="note">Top-1 = model's first choice is correct. Top-3/5 = correct is in the top 3 or 5 predictions.
  High Top-5 but low Top-1 means the model knows the sign is one of a few options but can't discriminate confidently.</div>

  <h3>Confidence Calibration</h3>
  <p class="desc"><span style="color:#3498db">■ Blue = actual accuracy</span> &nbsp;
  <span style="color:#e74c3c">■ Red = avg confidence</span> — bars should be equal height in a well-calibrated model.</p>
  <svg width="780" height="200" style="overflow:visible">
    <line x1="45" y1="8" x2="45" y2="162" stroke="#ddd"/>
    <line x1="45" y1="162" x2="760" y2="162" stroke="#ddd"/>
    {cal_svg}
  </svg>

  <h3>Per-Class Accuracy — Worst First</h3>
  <input type="text" id="s1" onkeyup="ft('s1','ct')" placeholder="Filter by sign…" style="margin-bottom:10px">
  <table><thead><tr>
    <th>#</th><th>Sign</th><th>Top-1</th><th>Top-3</th><th>Top-5</th>
    <th>Val Samples</th><th>Conf (correct)</th><th>Conf (wrong)</th>
  </tr></thead><tbody id="ct">{class_rows}</tbody></table>

  <h3 style="margin-top:24px">Top 30 Confusion Pairs</h3>
  <p class="desc">Which signs the model most often confuses. High-count pairs need contrastive training examples.</p>
  <table style="max-width:500px"><thead><tr>
    <th>True</th><th></th><th>Predicted As</th><th>Count</th>
  </tr></thead><tbody>{conf_rows}</tbody></table>
</div>

{fs_section}

<h2>Section 3 — Continuous Feed Pipeline Simulation</h2>
<div class="sec">
  <div class="cards">
    <div class="card"><div class="cv" style="color:{'#27ae60' if pipeline_acc>.6 else '#e74c3c'}">{pct(pipeline_acc)}</div><div class="cl">Correct Committed</div></div>
    <div class="card"><div class="cv" style="color:#e74c3c">{pct(false_pos_rate)}</div><div class="cl">Wrong Committed</div></div>
    <div class="card"><div class="cv" style="color:#e67e22">{pct(silence_rate)}</div><div class="cl">Nothing Committed</div></div>
    <div class="card"><div class="cv" style="color:#3498db">{avg_commit_f:.0f}</div><div class="cl">Avg Commit Frame</div></div>
    <div class="card"><div class="cv" style="color:#9b59b6">{avg_live_f:.0f}</div><div class="cl">Avg Live Frame</div></div>
    <div class="card"><div class="cv" style="color:#95a5a6">{n_sim}</div><div class="cl">Clips Simulated</div></div>
  </div>
  <div class="note">
    This section runs the <strong>complete pipeline</strong> exactly as <code>run_captions.py</code> does:
    frames fed one-by-one into a sliding buffer, both model heads called every frame,
    <code>format_token()</code> and <code>CaptionState.update()</code> applied.
    "Correct committed" = the right sign eventually locked in.
    "Wrong committed" = a different sign locked in.
    "Nothing committed" = the model never crossed the threshold for enough frames.
    <br><br>
    Note: pipeline accuracy will always be lower than batch Top-1 because the sliding
    window may not align perfectly with the sign's peak frames. This is the real-world number.
  </div>

  <h3>Worst-Performing Signs in Pipeline (20 shown)</h3>
  <table style="max-width:600px"><thead><tr>
    <th>Sign</th><th>Pipeline Acc</th><th>n clips</th>
    <th>Wrong Commits</th><th>Avg Commit Frame</th>
  </tr></thead><tbody>{sim_rows}</tbody></table>
</div>

<h2>Section 4 — Confidence Threshold Sweep</h2>
<div class="sec">
  <p class="desc">How changing <code>CONFIDENCE_THRESH</code> in <code>run_captions.py</code> affects the precision/recall tradeoff.
  Current value is <strong>0.35</strong> (highlighted).
  Raising it makes the model more careful but misses more signs.
  Lowering it catches more signs but produces more false commits.</p>
  <table style="max-width:580px"><thead><tr>
    <th>Threshold</th><th>Precision</th><th>Recall</th>
    <th>False Positives</th><th>Missed Signs</th><th>Total Fires</th>
  </tr></thead><tbody>{thr_rows}</tbody></table>
</div>

<h2>Section 5 — Silence Rejection</h2>
<div class="sec">
  <div class="cards">
    <div class="card"><div class="cv" style="color:{'#27ae60' if false_fires_silence==0 else '#e74c3c'}">{false_fires_silence}</div><div class="cl">False Commits (silence)</div></div>
  </div>
  <p class="desc">128 frames of zero landmarks (signer not signing) were fed through the pipeline.
  Any commits here are false positives caused by noise or model bias at rest.</p>

  <h3>Duplicate Suppression Test</h3>
  <p class="desc">Same sign fed twice with a {SILENCE_FRAMES}-frame gap between repetitions.
  Expected: each sign committed at most twice (once per occurrence).
  Extra commits indicate the hold-frame deduplication is not suppressing repeats correctly.</p>
  <table style="max-width:500px"><thead><tr>
    <th>Sign</th><th>Times Committed</th><th>Committed Tokens</th><th>Status</th>
  </tr></thead><tbody>{dup_rows}</tbody></table>
</div>

<h2>Priority Improvement Targets</h2>
<div class="sec">
  <p class="desc">Signs that fail the full pipeline (committed correctly &lt;30% of the time in simulation)
  AND have low batch top-1 accuracy. These are the highest-value targets:
  fixing them would directly improve the captions users see.</p>
  <table style="max-width:480px"><thead><tr>
    <th>Sign</th><th>Pipeline Acc</th><th>Batch Top-1</th><th>Wrong Commits</th>
  </tr></thead><tbody>{priority_rows}</tbody></table>
  {'<p style="color:#888;font-style:italic">No signs below 30% pipeline accuracy — strong result.</p>' if not priority_rows else ''}
</div>

<div style="color:#bbb;font-size:.77em;text-align:center;padding:30px 0">
  ASL Full Pipeline Validation · validate_kaggle.ipynb
</div>
</body></html>
"""

REPORT.write_text(html)
print(f'Report saved: {REPORT}  ({REPORT.stat().st_size/1e3:.0f} KB)')
print('Download from the Output tab and open in any browser.')

Report saved: /kaggle/working/validation_report.html  (179 KB)
Download from the Output tab and open in any browser.
